# 📊 Thesis Figure Generator

**Purpose:** Generate individual figures for Chapter 4 as requested by adviser

**Figures Generated:**
- Figure 4.2.3: Training Efficiency Comparison (PPO vs DQN)
- Figure 4.3.1: Overall Metrics Comparison
- Figure 4.3.2: Thin Cloud Recall Comparison
- Figure 4.3.3-4.3.5: Individual Patch Comparisons (enlarged)

**Instructions:** Run all cells in order.

## 1️⃣ Setup Environment

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone/update repository
import os
if os.path.exists('/content/thesis-cloud-rl'):
    %cd /content/thesis-cloud-rl
    !git pull origin master
else:
    !git clone https://github.com/Usernamenisiya/thesis-cloud-rl.git
    %cd /content/thesis-cloud-rl

print("✅ Repository ready")

In [ ]:
# Install dependencies
!pip install -q rasterio s2cloudless stable-baselines3 gymnasium scikit-learn

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import glob
import rasterio

print("✅ Dependencies installed")

In [ ]:
# Setup paths
BASE_DIR = Path('/content/drive/MyDrive/Colab_Data')
OUTPUT_DIR = BASE_DIR / 'thesis_figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = BASE_DIR / 'cloudsen12_processed_1000'
DQN_MODEL_PATH = BASE_DIR / 'dqn_thin_cloud' / 'dqn_thin_cloud_100000_steps.zip'
PPO_MODEL_PATH = BASE_DIR / 'thin_cloud_v2'  # Will find latest checkpoint

print(f"📁 Output directory: {OUTPUT_DIR}")
print(f"📁 Data directory: {DATA_DIR}")
print(f"✅ Paths configured")

## 2️⃣ Generate Bar Charts (No Data Required)

In [ ]:
# Figure 4.3.1: Overall Metrics Comparison

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
cnn_values = [78.49, 81.20, 73.79, 72.03]
ppo_values = [79.90, 80.00, 74.20, 72.79]
dqn_values = [80.89, 79.70, 76.70, 73.38]

x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 7))

bars1 = ax.bar(x - width, cnn_values, width, label='CNN Baseline', 
               color='#3498db', edgecolor='black', linewidth=1.2)
bars2 = ax.bar(x, ppo_values, width, label='PPO (720k steps)', 
               color='#e74c3c', edgecolor='black', linewidth=1.2)
bars3 = ax.bar(x + width, dqn_values, width, label='DQN (100k steps)', 
               color='#2ecc71', edgecolor='black', linewidth=1.2)

ax.set_ylabel('Percentage (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Performance Metrics', fontsize=14, fontweight='bold')
ax.set_title('Figure 4.3.1: Overall Performance Metrics Comparison\nCNN Baseline vs PPO vs DQN', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12, fontweight='bold')
ax.legend(loc='upper right', fontsize=11, framealpha=0.9)
ax.set_ylim(65, 95)

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.1f}%',
                   xy=(bar.get_x() + bar.get_width() / 2, height),
                   xytext=(0, 3), textcoords="offset points",
                   ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)
plt.tight_layout()

save_path = OUTPUT_DIR / 'Figure_4_3_1_Overall_Metrics.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"✅ Saved: {save_path}")
plt.show()

In [ ]:
# Figure 4.3.2: Thin Cloud Recall Comparison

models = ['CNN Baseline', 'PPO\n(720k steps)', 'DQN\n(100k steps)']
thin_recall = [63.28, 71.64, 77.00]
colors = ['#3498db', '#e74c3c', '#2ecc71']

fig, ax = plt.subplots(figsize=(10, 8))

bars = ax.bar(models, thin_recall, color=colors, edgecolor='black', linewidth=2, width=0.6)

ax.set_ylabel('Thin Cloud Recall (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Detection Method', fontsize=14, fontweight='bold')
ax.set_title('Figure 4.3.2: Thin Cloud Recall Comparison\n(Key Research Metric)', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_ylim(50, 85)

# Add value labels
for bar, val in zip(bars, thin_recall):
    ax.annotate(f'{val:.2f}%',
               xy=(bar.get_x() + bar.get_width() / 2, val),
               xytext=(0, 5), textcoords="offset points",
               ha='center', va='bottom', fontsize=16, fontweight='bold')

# Improvement annotations
ax.text(0.5, 67.5, '+8.36%', ha='center', fontsize=11, fontweight='bold', color='#e74c3c')
ax.text(1.0, 70.0, '+13.72%', ha='center', fontsize=11, fontweight='bold', color='#2ecc71')

ax.axhline(y=63.28, color='#3498db', linestyle='--', linewidth=2, alpha=0.7, label='Baseline: 63.28%')
ax.axhline(y=77.00, color='#2ecc71', linestyle='--', linewidth=2, alpha=0.7, label='Target Achieved: 77.00%')

ax.legend(loc='lower right', fontsize=10)
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)

plt.tight_layout()

save_path = OUTPUT_DIR / 'Figure_4_3_2_Thin_Cloud_Recall.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"✅ Saved: {save_path}")
plt.show()

In [ ]:
# Figure 4.2.3: Training Efficiency Comparison

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Training steps
models_eff = ['PPO', 'DQN']
steps = [720000, 100000]
colors_eff = ['#e74c3c', '#2ecc71']

bars1 = axes[0].bar(models_eff, steps, color=colors_eff, edgecolor='black', linewidth=2, width=0.5)
axes[0].set_ylabel('Training Steps', fontsize=13, fontweight='bold')
axes[0].set_title('(a) Training Steps Required', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, 800000)

for bar, val in zip(bars1, steps):
    axes[0].annotate(f'{val:,}',
                    xy=(bar.get_x() + bar.get_width() / 2, val),
                    xytext=(0, 5), textcoords="offset points",
                    ha='center', fontsize=14, fontweight='bold')

axes[0].annotate('7× fewer steps!', xy=(1, 100000), xytext=(0.5, 400000),
                fontsize=12, fontweight='bold', color='#2ecc71',
                arrowprops=dict(arrowstyle='->', color='#2ecc71', lw=2))
axes[0].yaxis.grid(True, linestyle='--', alpha=0.7)

# Right: Performance
thin_recall_eff = [71.64, 77.00]

bars2 = axes[1].bar(models_eff, thin_recall_eff, color=colors_eff, edgecolor='black', linewidth=2, width=0.5)
axes[1].set_ylabel('Thin Cloud Recall (%)', fontsize=13, fontweight='bold')
axes[1].set_title('(b) Performance Achieved', fontsize=14, fontweight='bold')
axes[1].set_ylim(60, 85)

for bar, val in zip(bars2, thin_recall_eff):
    axes[1].annotate(f'{val:.2f}%',
                    xy=(bar.get_x() + bar.get_width() / 2, val),
                    xytext=(0, 5), textcoords="offset points",
                    ha='center', fontsize=14, fontweight='bold')

axes[1].annotate('+5.36% better!', xy=(1, 77.00), xytext=(0.5, 82),
                fontsize=12, fontweight='bold', color='#2ecc71',
                arrowprops=dict(arrowstyle='->', color='#2ecc71', lw=2))
axes[1].yaxis.grid(True, linestyle='--', alpha=0.7)

fig.suptitle('Figure 4.2.3: Training Efficiency Comparison (PPO vs DQN)\n'
             'DQN achieves better results with 7× fewer training steps',
             fontsize=15, fontweight='bold', y=1.02)

plt.tight_layout()

save_path = OUTPUT_DIR / 'Figure_4_2_3_Training_Efficiency.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"✅ Saved: {save_path}")
plt.show()

## 3️⃣ Load Data and Models for Patch Figures

In [ ]:
# Load test data
print("📊 Loading test data...")

image_files = sorted(glob.glob(f'{DATA_DIR}/*_image.tif'))
mask_files = sorted(glob.glob(f'{DATA_DIR}/*_mask.tif'))

# Use test set (last 200 patches)
split_idx = int(0.8 * len(image_files))
test_images = image_files[split_idx:]
test_masks = mask_files[split_idx:]

print(f"✅ Found {len(test_images)} test images")

# Load a few patches
def load_patch(img_path, mask_path):
    with rasterio.open(img_path) as src:
        patch = src.read()
        patch = np.transpose(patch, (1, 2, 0))
    with rasterio.open(mask_path) as src:
        label = src.read(1)
    return patch, label

# Select patches with good thin cloud examples
selected_indices = [0, 1, 4]  # Adjust if needed
patches = []
labels = []

for idx in selected_indices:
    patch, label = load_patch(test_images[idx], test_masks[idx])
    patches.append(patch)
    labels.append(label)
    thin_pct = (label == 2).sum() / max((label > 0).sum(), 1) * 100
    print(f"  Patch {idx}: {thin_pct:.1f}% thin clouds")

print(f"\n✅ Loaded {len(patches)} patches for visualization")

In [ ]:
# Load models
print("🤖 Loading models...")

from stable_baselines3 import DQN, PPO
from s2cloudless import S2PixelCloudDetector

# Load DQN model
dqn_model = DQN.load(str(DQN_MODEL_PATH))
print(f"✅ DQN model loaded from: {DQN_MODEL_PATH}")

# Initialize CNN baseline
cloud_detector = S2PixelCloudDetector(threshold=0.4, all_bands=True, average_over=4)
print("✅ CNN baseline (s2cloudless) initialized")

In [ ]:
# Import RL environment
import sys
sys.path.append('/content/thesis-cloud-rl')

from rl_thin_cloud_environment_discrete import ThinCloudDetectionEnvDiscrete

print("✅ RL environment imported")

In [ ]:
# Apply CNN baseline and DQN to selected patches
print("🔄 Processing patches...")

cnn_masks = []
dqn_masks = []
cnn_probs = []

for i, (patch, label) in enumerate(zip(patches, labels)):
    print(f"  Processing patch {i+1}/{len(patches)}...")
    
    # CNN baseline
    patch_normalized = patch / 10000.0 if patch.max() > 1.0 else patch
    patch_batched = patch_normalized[np.newaxis, ...]
    cnn_prob = cloud_detector.get_cloud_probability_maps(patch_batched)[0]
    cnn_mask = (cnn_prob > 0.5).astype(np.uint8)
    
    cnn_masks.append(cnn_mask)
    cnn_probs.append(cnn_prob)
    
    # DQN refinement
    gt_binary = (label > 0).astype(np.uint8)
    env = ThinCloudDetectionEnvDiscrete(patch, cnn_prob, gt_binary)
    
    prediction = np.zeros_like(cnn_prob, dtype=np.uint8)
    ps = env.patch_size
    
    obs, _ = env.reset()
    done = False
    
    while not done:
        action, _ = dqn_model.predict(obs, deterministic=True)
        
        threshold_idx = action // 3
        boost_idx = action % 3
        threshold_delta = env.THRESHOLD_OPTIONS[threshold_idx]
        thin_boost = env.BOOST_OPTIONS[boost_idx]
        
        row, col = env.current_pos
        cnn_patch = cnn_prob[row:row+ps, col:col+ps].copy()
        thin_indicator = env.thin_cloud_indicator[row:row+ps, col:col+ps]
        
        boosted_prob = np.clip(cnn_patch + thin_indicator * thin_boost, 0, 1)
        prediction[row:row+ps, col:col+ps] = (boosted_prob > (0.5 + threshold_delta)).astype(np.uint8)
        
        obs, reward, done, truncated, info = env.step(action)
        done = done or truncated
    
    dqn_masks.append(prediction)

print("\n✅ All patches processed")

## 4️⃣ Generate Individual Patch Figures

In [ ]:
# Helper functions

def create_rgb_image(patch):
    """Create RGB image from patch."""
    rgb = patch[:, :, [2, 1, 0]]
    rgb = np.clip(rgb / 3000.0, 0, 1)
    rgb = np.power(rgb, 0.7)
    return rgb

def create_error_overlay(pred, gt, thin_mask=None):
    """Create color-coded error overlay."""
    overlay = np.zeros((*pred.shape, 3), dtype=np.float32)
    
    tp = (pred == 1) & (gt == 1)
    fn = (pred == 0) & (gt == 1)
    fp = (pred == 1) & (gt == 0)
    tn = (pred == 0) & (gt == 0)
    
    overlay[tp] = [0.2, 0.8, 0.2]   # Green for TP
    overlay[fn] = [0.9, 0.2, 0.2]   # Red for FN
    overlay[fp] = [0.2, 0.4, 0.9]   # Blue for FP
    overlay[tn] = [0.85, 0.85, 0.85] # Light gray for TN
    
    if thin_mask is not None:
        thin_tp = tp & thin_mask
        overlay[thin_tp] = [0.0, 0.9, 0.9]  # Cyan for thin cloud TP
    
    return overlay

print("✅ Helper functions defined")

In [ ]:
# Generate Figure 4.3.3, 4.3.4, 4.3.5 - Individual Patch Comparisons

for idx in range(len(patches)):
    patch = patches[idx]
    label = labels[idx]
    cnn_mask = cnn_masks[idx]
    dqn_mask = dqn_masks[idx]
    
    gt_binary = (label > 0).astype(np.uint8)
    thin_mask = (label == 2)
    
    # Calculate thin cloud recall
    thin_total = thin_mask.sum()
    if thin_total > 0:
        cnn_thin_recall = (cnn_mask[thin_mask].sum() / thin_total) * 100
        dqn_thin_recall = (dqn_mask[thin_mask].sum() / thin_total) * 100
    else:
        cnn_thin_recall = 0.0
        dqn_thin_recall = 0.0
    
    # Create 4-panel figure
    fig, axes = plt.subplots(2, 2, figsize=(14, 14))
    
    # Panel 1: RGB Image
    rgb = create_rgb_image(patch)
    axes[0, 0].imshow(rgb)
    axes[0, 0].set_title('(a) Original RGB Image', fontsize=14, fontweight='bold')
    axes[0, 0].axis('off')
    
    # Panel 2: Ground Truth
    gt_display = np.zeros((*label.shape, 3))
    gt_display[label == 0] = [0.85, 0.85, 0.85]  # Clear
    gt_display[label == 1] = [0.8, 0.2, 0.2]      # Thick cloud
    gt_display[label == 2] = [0.0, 0.8, 0.8]      # Thin cloud
    gt_display[label == 3] = [0.3, 0.3, 0.3]      # Shadow
    
    axes[0, 1].imshow(gt_display)
    axes[0, 1].set_title('(b) Ground Truth Labels', fontsize=14, fontweight='bold')
    axes[0, 1].axis('off')
    
    legend_elements = [
        mpatches.Patch(facecolor=[0.85, 0.85, 0.85], edgecolor='black', label='Clear Sky'),
        mpatches.Patch(facecolor=[0.8, 0.2, 0.2], edgecolor='black', label='Thick Cloud'),
        mpatches.Patch(facecolor=[0.0, 0.8, 0.8], edgecolor='black', label='Thin Cloud'),
        mpatches.Patch(facecolor=[0.3, 0.3, 0.3], edgecolor='black', label='Shadow'),
    ]
    axes[0, 1].legend(handles=legend_elements, loc='lower right', fontsize=9)
    
    # Panel 3: CNN Baseline
    cnn_overlay = create_error_overlay(cnn_mask, gt_binary, thin_mask)
    axes[1, 0].imshow(cnn_overlay)
    axes[1, 0].set_title(f'(c) CNN Baseline\nThin Cloud Recall: {cnn_thin_recall:.1f}%', 
                         fontsize=14, fontweight='bold')
    axes[1, 0].axis('off')
    
    # Panel 4: DQN Refined
    dqn_overlay = create_error_overlay(dqn_mask, gt_binary, thin_mask)
    axes[1, 1].imshow(dqn_overlay)
    axes[1, 1].set_title(f'(d) DQN Refined\nThin Cloud Recall: {dqn_thin_recall:.1f}%', 
                         fontsize=14, fontweight='bold')
    axes[1, 1].axis('off')
    
    # Common legend
    error_legend = [
        mpatches.Patch(facecolor=[0.2, 0.8, 0.2], edgecolor='black', label='True Positive'),
        mpatches.Patch(facecolor=[0.9, 0.2, 0.2], edgecolor='black', label='False Negative (Missed)'),
        mpatches.Patch(facecolor=[0.2, 0.4, 0.9], edgecolor='black', label='False Positive'),
        mpatches.Patch(facecolor=[0.0, 0.9, 0.9], edgecolor='black', label='Thin Cloud Detected'),
    ]
    fig.legend(handles=error_legend, loc='lower center', ncol=4, fontsize=11, 
               bbox_to_anchor=(0.5, 0.02))
    
    # Main title
    improvement = dqn_thin_recall - cnn_thin_recall
    fig.suptitle(f'Figure 4.3.{idx+3}: Patch #{idx+1} Detailed Comparison\n'
                 f'Improvement: +{improvement:.1f}% Thin Cloud Recovery',
                 fontsize=16, fontweight='bold', y=0.98)
    
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    
    save_path = OUTPUT_DIR / f'Figure_4_3_{idx+3}_Patch_{idx+1}_Comparison.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"✅ Saved: {save_path}")
    
    plt.show()
    plt.close()

## 5️⃣ Summary

In [ ]:
# List all generated figures
print("="*60)
print("📊 THESIS FIGURES GENERATED SUCCESSFULLY!")
print("="*60)
print(f"\n📁 Output directory: {OUTPUT_DIR}")
print("\n📄 Files generated:")

for f in sorted(OUTPUT_DIR.glob('Figure_*.png')):
    print(f"   ✅ {f.name}")

print("\n" + "="*60)
print("Copy these figures to your thesis manuscript!")
print("="*60)